# Conventional augmentation experiment
Run on Kaggle with Skin Cancer ISIC, ISIC 2019 Training Ground Truth, and ISIC 2019 Metadata attached and a GPU enabled. The single code cell rebuilds the verified split and checks its frozen membership fingerprint before training.

The CNN architecture, class weights, learning rate, batch size, epoch budget and early-stopping rule match the recovered baseline. Training augmentation: horizontal/vertical flips, rotation up to 0.05 turns (18 degrees), zoom factor 0.10, reflection filling. Validation uses original resized images; test images are not evaluated.

The comparison is an initial single-run experiment. The same seed does not guarantee identical random trajectories or fully deterministic GPU results; repeated seeds are needed for a robust conclusion.

Keras documentation: https://keras.io/api/layers/preprocessing_layers/image_augmentation/random_rotation/ and https://keras.io/api/layers/preprocessing_layers/image_augmentation/random_zoom/


Import the split dataset and inspect

In [24]:
#Import all necessary libraries
from pathlib import Path
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
import pandas as pd
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, balanced_accuracy_score
#Import all necessary libraries
from pathlib import Path
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from hashlib import sha256
import json, gc, zipfile
import numpy as np
import tensorflow as tf
from sklearn.metrics import (classification_report, confusion_matrix,
    accuracy_score, f1_score, balanced_accuracy_score, roc_auc_score,
    average_precision_score)
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    auc
)

In [25]:
MAPPING = {
    **dict.fromkeys(['NV', 'BKL', 'DF', 'VASC'], 'benign'),
    **dict.fromkeys(['MEL', 'BCC', 'SCC', 'AK'], 'malignant')
}

fingerprint_path = Path.cwd().parent / "artifacts" / "dataset_fingerprint.json"

with open(fingerprint_path) as f:
    fingerprint_data = json.load(f)
fingerprint = fingerprint_data["fingerprint"]

In [26]:
data_path = Path.cwd().parent / "artifacts" / "isic2019_binary_split.csv"
out = Path.cwd().parent /'all_augmentations_results'
out.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(data_path)
print(df.head())
print(df.columns)
print(df["split"].value_counts())
print(df.groupby(["split", "label"]).size().unstack(fill_value=0))

       image_id                                               path      label  \
0  ISIC_0000001   e:\Nová složka\input\Test\nevus\ISIC_0000001.jpg     benign   
1  ISIC_0000002  e:\Nová složka\input\Test\melanoma\ISIC_000000...  malignant   
2  ISIC_0000006   e:\Nová složka\input\Test\nevus\ISIC_0000006.jpg     benign   
3  ISIC_0000007   e:\Nová složka\input\Test\nevus\ISIC_0000007.jpg     benign   
4  ISIC_0000008   e:\Nová složka\input\Test\nevus\ISIC_0000008.jpg     benign   

  lesion_id               group  split original_label  target  
0       NaN  image:ISIC_0000001  train             NV       0  
1       NaN  image:ISIC_0000002  train            MEL       1  
2       NaN  image:ISIC_0000006  train             NV       0  
3       NaN  image:ISIC_0000007  train             NV       0  
4       NaN  image:ISIC_0000008  train             NV       0  
Index(['image_id', 'path', 'label', 'lesion_id', 'group', 'split',
       'original_label', 'target'],
      dtype='object')
spli

In [27]:
train_df = df[df["split"] == "train"].copy()
val_df   = df[df["split"] == "val"].copy()
test_df  = df[df["split"] == "test"].copy()

In [28]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

# Create a consistent mapping of class names to integer IDs
class_names = sorted(df["label"].unique().tolist())
class_to_id = {
    "benign": 0,
    "malignant": 1
}

assert set(df["label"].unique()) == set(class_to_id.keys())

class_names = ["benign", "malignant"]

def make_dataset(frame, training=False, seed=42):

    paths = frame["path"].astype(str).values
    labels = frame["label"].map(class_to_id).astype("float32").values

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    def load_image(path, label):
        image = tf.io.read_file(path)
        image = tf.io.decode_image(
            image, channels=3, expand_animations=False
        )
        image.set_shape([None, None, 3])
        image = tf.image.resize(image, IMG_SIZE)
        image = tf.cast(image, tf.float32) / 255.0

        return image, label

    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)

    if training:
        ds = ds.shuffle(
            buffer_size=len(frame),
            seed=seed,
            reshuffle_each_iteration=True
        )

    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)

In [29]:
y_train = train_df["label"].map(class_to_id).to_numpy()
classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = {
    int(cls): float(weight)
    for cls, weight in zip(classes, weights)
}

print(class_weights)
class_to_num = class_to_id
num_to_class = {v: k for k, v in class_to_num.items()}

{0: 0.9814126394052045, 1: 1.0193050193050193}


CNN

In [30]:
def build_cnn(filters, kernel_size, dense_units, dropout_rate):

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(128, 128,3)),

        tf.keras.layers.Conv2D(filters[0], kernel_size, activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Conv2D(filters[1], kernel_size, activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Conv2D(filters[2], kernel_size, activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Conv2D(filters[3], kernel_size, activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Flatten(),

        tf.keras.layers.Dense(dense_units, activation='relu'),
        tf.keras.layers.Dropout(dropout_rate),

        tf.keras.layers.Dense(1, activation='sigmoid')
    ])

    return model

In [31]:
def build_augmentation(seed):

    return tf.keras.Sequential([
        tf.keras.layers.RandomFlip(
            "horizontal_and_vertical", seed=seed
        ),

        tf.keras.layers.RandomRotation(
            0.05, fill_mode="reflect", seed=seed + 1
        ),

        tf.keras.layers.RandomZoom(
            0.10, fill_mode="reflect", seed=seed + 2
        ),

        tf.keras.layers.RandomTranslation(
            height_factor=0.10,
            width_factor=0.10,
            fill_mode="reflect",
            seed=seed + 3
        ),

        tf.keras.layers.RandomCrop(
            height=110, width=110, seed=seed + 4
        ),

        tf.keras.layers.Resizing(128, 128),

        tf.keras.layers.RandomBrightness(
            0.15, value_range=(0, 1), seed=seed + 5
        ),

        tf.keras.layers.RandomContrast(
            0.15, seed=seed + 6
        )

    ], name=f"conventional_augmentation_seed_{seed}")


In [32]:
best_filters = (32, 64, 128, 256)
best_kernel_size = 5
best_dense_units = 128
best_dropout_rate = 0.45
best_learning_rate = 0.0001

#Run the model 
results = []
for seed in [42,43,44]:
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)
    model = build_cnn(
        filters=best_filters,
        kernel_size=best_kernel_size,
        dense_units=best_dense_units,
        dropout_rate=best_dropout_rate
    )
    init_hash = sha256(b''.join(w.tobytes() for w in model.get_weights())).hexdigest()
    # Preserve initial arrays for future paired binary augmentation comparisons.
    np.savez_compressed(out/f'initial_weights_seed{seed}.npz', *model.get_weights())
    model.compile(optimizer=tf.keras.optimizers.Adam(
        learning_rate=best_learning_rate),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])

    run = f'binary_aug_seed{seed}'
    checkpoint = out/f'{run}.keras'

    # Build augmentation pipeline
    augmentation = build_augmentation(seed)

    # Create training dataset
    train_ds = make_dataset(
        train_df,
        training=True,
        seed=seed
    )

    # Apply augmentation to training images only
    train_ds = train_ds.map(
        lambda images, labels: (
            augmentation(images, training=True),
            labels
        ),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    # Create validation dataset WITHOUT augmentation
    val_ds = make_dataset(
        val_df,
        training=False,
        seed=seed
    )
    
    history = model.fit(train_ds, validation_data=val_ds, epochs=25,
        class_weight=class_weights, callbacks=[
            tf.keras.callbacks.ModelCheckpoint(str(checkpoint),
                monitor='val_loss',save_best_only=True),
            tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=5,
                                             restore_best_weights=True)], verbose=2)
    best = tf.keras.models.load_model(checkpoint)
    probability = best.predict(val_ds,verbose=0).reshape(-1)
    y = val_df["label"].map(class_to_id).to_numpy(dtype=int)
    if len(np.unique(y)) != 2:
        raise ValueError(
        "Validation set must contain both benign and malignant classes.")
    pred = (probability >= .5).astype(int)
    tn,fp,fn,tp = confusion_matrix(y,pred,labels=[0,1]).ravel()
    row = dict(seed=seed,condition='binary_aug',
        epochs_trained=len(history.history['loss']),
        selected_epoch=int(np.argmin(history.history['val_loss'])+1),
        validation_loss=float(min(history.history['val_loss'])),
        accuracy=float(accuracy_score(y,pred)),
        macro_f1=float(f1_score(y,pred,labels=[0,1],average='macro',zero_division=0)),
        balanced_accuracy=float(balanced_accuracy_score(y,pred)),
        malignant_sensitivity=float(tp / (tp + fn)) if (tp + fn) > 0 else 0.0,specificity=float(tn / (tn + fp)) if (tn + fp) > 0 else 0.0,
        roc_auc=float(roc_auc_score(y,probability)),
        average_precision=float(average_precision_score(y,probability)),
        threshold=.5,initial_weights_sha256=init_hash,
        split_fingerprint=fingerprint,test_evaluated=False)
    report=classification_report(y,pred,labels=[0,1],
        target_names=['benign','malignant'],output_dict=True,zero_division=0)
    (out/f'{run}_metrics.json').write_text(json.dumps(
        dict(**row,classification_report=report),indent=2))
    pd.DataFrame(history.history).assign(
        epoch=lambda d: np.arange(1,len(d)+1)).to_csv(out/f'{run}_history.csv',index=False)
    predictions=val_df[['image_id','original_label','label','group']].copy()
    predictions['malignant_probability']=probability
    predictions['prediction']=pred
    predictions.to_csv(out/f'{run}_validation_predictions.csv',index=False)
    pd.DataFrame(confusion_matrix(y,pred,labels=[0,1]),
        index=['benign','malignant'],columns=['benign','malignant']).to_csv(
            out/f'{run}_confusion_matrix.csv')
# Save confusion matrix as PNG
    cm = confusion_matrix(y, pred, labels=[0, 1])

    plt.figure(figsize=(7, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=['Benign', 'Malignant'],
        yticklabels=['Benign', 'Malignant'],
        annot_kws={'size': 16}
    )

    plt.xlabel('Predicted label', fontsize=12)
    plt.ylabel('True label', fontsize=12)
    plt.title(f'Confusion Matrix - {run}', fontsize=14)
    plt.tight_layout()

    plt.savefig(
        out/f'{run}_confusion_matrix.png',
        dpi=300,
        bbox_inches='tight'
    )
    plt.close()
    # Generate ROC curve
    fpr, tpr, thresholds = roc_curve(y, probability)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(7, 6))

    plt.plot(
        fpr,
        tpr,
        color='darkorange',
        lw=2,
        label=f'ROC curve (AUC = {roc_auc:.3f})'
    )

    plt.plot(
        [0, 1],
        [0, 1],
        color='navy',
        lw=2,
        linestyle='--',
        label='Random classifier'
    )

    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])

    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title(f'ROC Curve - {run}', fontsize=14)

    plt.legend(loc='lower right')
    plt.grid(alpha=0.3)
    plt.tight_layout()

    plt.savefig(
        out/f'{run}_ROC_curve.png',
        dpi=300,
        bbox_inches='tight'
    )
    plt.close()    
    results.append(row)
    pd.DataFrame(results).to_csv(out/'binary_augmented_results.csv',index=False)
    # Checkpoint each seed's complete evidence before moving to the next run.
    
    # Archive lightweight experimental evidence
    archive_path = out / f'{run}_artifacts.zip'

    files_to_archive = [
            out / f'{run}_metrics.json',
            out / f'{run}_history.csv',
            out / f'{run}_validation_predictions.csv',
            out / f'{run}_confusion_matrix.csv',
            out / f'{run}_confusion_matrix.png',
            out / f'{run}_ROC_curve.png',
            out / f'initial_weights_seed{seed}.npz',
            checkpoint,
            data_path,
            out / 'binary_augmented_results.csv'
        ]

    with zipfile.ZipFile(
        archive_path,
        'w',
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=6
    ) as archive:

        for p in files_to_archive:
            if p.exists():
                archive.write(p, p.name)
            else:
                print(f'Warning: missing file {p}')

    print(f'Archive saved: {archive_path}')
    print(f'Archive size: {archive_path.stat().st_size / (1024**2):.2f} MB')

    # Model and initial weights remain saved separately
    print(f'Model saved: {checkpoint}')
    print(f'Initial weights saved: initial_weights_seed{seed}.npz')

    print('BINARY_RUN_COMPLETE', json.dumps(row), flush=True)

    del model, best, train_ds, val_ds
    gc.collect()

Epoch 1/25
33/33 - 15s - 440ms/step - accuracy: 0.5066 - auc: 0.4918 - loss: 0.6973 - val_accuracy: 0.4929 - val_auc: 0.5858 - val_loss: 0.6935
Epoch 2/25
33/33 - 8s - 228ms/step - accuracy: 0.4962 - auc: 0.5127 - loss: 0.6937 - val_accuracy: 0.5071 - val_auc: 0.4925 - val_loss: 0.6929
Epoch 3/25
33/33 - 7s - 219ms/step - accuracy: 0.4962 - auc: 0.4986 - loss: 0.6932 - val_accuracy: 0.5071 - val_auc: 0.5791 - val_loss: 0.6924
Epoch 4/25
33/33 - 7s - 220ms/step - accuracy: 0.5379 - auc: 0.5274 - loss: 0.6932 - val_accuracy: 0.5694 - val_auc: 0.6195 - val_loss: 0.6912
Epoch 5/25
33/33 - 7s - 222ms/step - accuracy: 0.5417 - auc: 0.5542 - loss: 0.6911 - val_accuracy: 0.5807 - val_auc: 0.7014 - val_loss: 0.6846
Epoch 6/25
33/33 - 7s - 222ms/step - accuracy: 0.5492 - auc: 0.5908 - loss: 0.6842 - val_accuracy: 0.5921 - val_auc: 0.6447 - val_loss: 0.6744
Epoch 7/25
33/33 - 3s - 93ms/step - accuracy: 0.5890 - auc: 0.6109 - loss: 0.6787 - val_accuracy: 0.5836 - val_auc: 0.6624 - val_loss: 0.6744